# 02 · MCP 回放服务与连接

FinMCP-Bench 的真实工具在且慢的 MCP 服务上。为了**可复现、不花钱、不受数据变化影响**，这里起一个本地的 MCP 回放服务（`python -m jif.replay_server`，stdio 协议），用日志里记录的返回结果回答工具调用：

1. 当前任务里参数完全相同的调用 → `hit`
2. 其他任务里参数完全相同的调用 → `hit`
3. 同一工具、参数最接近的调用（重合度 ≥ 0.5，或我方请求的值大多被覆盖）→ `approx`
4. 否则 → `miss`，返回错误

FinMCP 没有提供工具的参数定义，服务从日志**反推**每个工具的参数（名称、类型、是否必填、取值示例）。

客户端 `McpConnection` 在后台线程里跑异步会话，所以在 notebook 里可以直接同步调用。**这一本不花钱。**

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import collections, json, time
import pandas as pd
from jif import finmcp
from jif.mcp_conn import McpConnection
pd.set_option("display.max_colwidth", 100)

t0 = time.time()
conn = McpConnection()
print(f"回放服务已启动（{time.time() - t0:.1f} 秒）")

In [ ]:
tools = conn.list_tools()
print("工具数：", len(tools))
display(pd.DataFrame([{"工具": t["name"], "参数": ", ".join(t["input_schema"].get("properties", {})),
                       "必填": ", ".join(t["input_schema"].get("required", [])), "说明": t["description"]} for t in tools]).head(15))
print(json.dumps(next(t for t in tools if t["name"] == "GetStrategyDetails")["input_schema"], ensure_ascii=False, indent=1))

## 回放一条标准轨迹

In [ ]:
tid = "multi_tool:30"
task = finmcp.get_task(tid)
print("问题：", finmcp.first_question(task))
conn.set_task(tid)
for c in finmcp.tool_calls(task):
    r = conn.call(c["name"], c["args"])
    print(f"{c['name']}({json.dumps(c['args'], ensure_ascii=False)}) → ok={r['ok']}，{len(r['text'])} 字符，{r['latency'] * 1000:.1f} ms")
pd.DataFrame(conn.replay_log())

## 自洽检查：所有多工具任务的标准调用都应当命中

In [ ]:
stats, t0 = collections.Counter(), time.time()
for tid, task in finmcp.iter_tasks(("multi_tool",)):
    conn.set_task(tid)
    for c in finmcp.tool_calls(task):
        conn.call(c["name"], c["args"])
    stats.update(e["status"] for e in conn.replay_log())
print(dict(stats), f"，用时 {time.time() - t0:.1f} 秒")

## 近似匹配与未命中

In [ ]:
conn.set_task("multi_tool:30")
conn.call("GetUserAssetHoldingDistribution", {"userId": "9b989f9b9d9b92", "poCodes": ["LONG_WIN", "ZH013136"]})  # 标准调用的子集
print(conn.call("SearchFunds", {"keyword": "一个不存在的关键词"})["text"])
print(conn.call("NoSuchTool", {})["text"])
pd.DataFrame(conn.replay_log())

In [ ]:
conn.close()